# Custom Cells: Synthetic Walkthrough

This optional lesson demonstrates live-cell adoption and a saved Python builder. It uses built-in `pas`, temporary workspaces, and no ACT/BMTool. Numeric choices are plumbing demonstrations, not scientific defaults. Run from SCP (or its notebook folder) with the normal `scp-py311` environment. See [Custom models](../docs/guides/custom_models.md).


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile
import numpy as np
from neuron import h

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "modules").is_dir() and (p / "run_pipeline.py").is_file())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from modules.model import wrap_cell, load_cell
from modules.simulation import SimulationOptions, SimulationSession
from modules.tuning import run_passive_protocol, run_active_protocol
from modules.setup import prepare_tune, preview_model_sources
from modules.simulation.result_loading import load_results

workspace = tempfile.TemporaryDirectory(prefix="scp_custom_cells_")
workspace_root = Path(workspace.name)
conditions = {"v_init_mV": -70.0, "celsius_C": 34.0}
smoke_config = {"conditions": conditions, "dt": 0.025, "tstop": 20.0,
                "plots_profile": "off",
                "iclamp": {"amp_nA": 0.025, "delay_ms": 5.0, "dur_ms": 10.0}}


## 1. Construct and adopt an existing cell

In a primary notebook, put this after environment setup, set `CELL_OVERRIDE` and `RUNTIME_SIM_CONFIG`, and select a new workspace. Keep associated native objects on your owner when using a larger cell.


In [ ]:
soma = h.Section(name="synthetic_body")
soma.L = soma.diam = 20.0
soma.insert("pas")
soma.g_pas, soma.e_pas = 0.0001, -70.0
CELL_OVERRIDE = wrap_cell(soma, cell_name="SyntheticLive")
RUNTIME_SIM_CONFIG = smoke_config
assert CELL_OVERRIDE.model is soma
assert CELL_OVERRIDE.all == (soma,)

protocol = {"stim_amp": 0.025, "stim_delay": 5.0, "stim_dur": 10.0,
            "h_tstop": 20.0, "h_dt": 0.025, "conditions": conditions}
passive = run_passive_protocol(cell=CELL_OVERRIDE, sim_params=protocol, sim_amps=[-25.0])
active_fi = run_active_protocol(cell=CELL_OVERRIDE, sim_params=protocol, sim_amps=[0.0, 25.0])
assert np.isfinite(passive["V"][-25.0]).all()
assert len(active_fi["T"][25.0]) == len(active_fi["V"][25.0]) == 801


## 2. Save a live-cell result

No Step 1 source recipe is needed. This creates only missing generic workspace configs and uses the same object. The saved summary is descriptive—not a checkpoint capable of recreating the live cell.


In [ ]:
live_tune = workspace_root / "live"
live_session = SimulationSession.from_cell(
    CELL_OVERRIDE, tune_dir=live_tune, sim_config=smoke_config,
    options=SimulationOptions(iclamp=True, force_save=True, output_stem="live_demo"),
)
live_result = live_session.run()
live_path = live_session.save()
assert live_session.cell is CELL_OVERRIDE
assert live_result["meta"]["model_provenance"]["session_only"]
assert not live_result["meta"]["model_provenance"]["construction_recipe_available"]
assert np.isfinite(load_results(live_path)["traces"]["V"]).all()
print("Live result is explicitly session-only:", live_path)


## 3. Register a saved Python builder

The committed fixture is a non-biological package with relative imports and an asset file. Copy mode snapshots the selected native folder without changing its source. Inspect the inventory before setup; reference mode would still retain a snapshot but load originals. Orig-to-tuned copying is a separate action.


In [ ]:
source_root = repo_root / "tests" / "fixtures" / "custom_models"
factory_tune = workspace_root / "factory"
factory_options = {
    "callable": "build_cell", "args": [], "kwargs": {"scale": 1.0},
    "section_map": {"soma": "body", "dend": "branch", "all": "whole_cell"},
}
source_options = {
    "mode": "copy", "source_root": str(source_root),
    "folders": ["package"], "working_directory": "package",
}
recipe = {
    "cell_name": "SyntheticFactory", "tune": "factory", "cell_loader": "python_factory",
    "paths": {"python_factory": "package/cell.py", "modfiles": None},
    "python_factory": factory_options, "model_source": source_options,
}
preview = preview_model_sources(factory_tune, recipe)
print("Preview:", preview["file_count"], "files,", preview["size_bytes"], "bytes")
setup = prepare_tune(
    tune_dir=factory_tune, cell_name="SyntheticFactory", tune_name="factory",
    source_type="existing", cell_loader="python_factory", loader_paths=recipe["paths"],
    loader_config={"python_factory": factory_options, "model_source": source_options},
    v_init_mV=-70.0, celsius_C=34.0, target_source_mode="none", do_synapse_configs=False,
)
assert setup["actions"]["validate"]["status"] == "ok"


## 4. Reconstruct in a fresh process

The CLI loads persisted code/config, not `CELL_OVERRIDE` or unsaved notebook edits. The short protocol is a temporary override. The same configured tune can be selected by SLURM. Restart when switching HOC definitions or loaded mechanism libraries.


In [ ]:
command = [
    sys.executable, str(repo_root / "run_pipeline.py"),
    "--tune-dir", str(factory_tune), "--iclamp", "--force-save",
    "--output-stem", "fresh_demo", "--sim-overrides-json", json.dumps(smoke_config),
]
worker = subprocess.run(command, cwd=repo_root,
                        env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1", MPLBACKEND="Agg"),
                        capture_output=True, text=True)
assert worker.returncode == 0, worker.stdout + worker.stderr
manifest = next((factory_tune / "output_data").glob("fresh_demo*/run_manifest.json"))
rebuilt = load_results(manifest)
assert rebuilt["meta"]["model_provenance"]["construction_recipe_available"]
assert np.isfinite(rebuilt["traces"]["V"]).all()
assert len(rebuilt["traces"]["V"]) == 801
print("Fresh-process builder result:", manifest)


## Next steps and cleanup

Use Step 2/3 with your scientific protocols and targets, not these smoke settings. Persist manual changes in copied native sources or builder arguments before worker execution. Step 6 can analyze saved traces without reconstruction; live inspection can differ from saved provenance. Step 7 materialization is dry-run-first and tune-local; a runtime-only object cannot be recovered.

The temporary workspace stays available while this notebook is open. Cleanup explicitly when finished; no production sources or existing SCP cells were edited.


In [ ]:
# Optional after inspecting the temporary results:
# workspace.cleanup()
